In [ ]:
# Local codebase import
from deps.ScanUtils import *

# Base defines 
PRJNAME = "recon"
ROOT = os.path.dirname(os.path.abspath("")) + "/" + PRJNAME
print(f"[*] Root is {ROOT}")

In [ ]:
# Main config
config = {
    "target": "10.129.206.254",
    # Main interface gateway for tools like nmap etc
    "iface": "tun0",
    # Testing scope
    "scope": [
        # "basic",
        "ports",
        "web_scan"
    ],
    # General info scan
    "basic_scan": {
        "subscans": [
            # "whois",
            "geo",
            # "dns-zt",
            "os-detect",
            "traceroute",
        ],
        "geodb": "assets/dbip-city-lite-2023-02.mmdb"
    },
    # Subconfig for ports scan from scope
    "port_scan": {
        # Scan top ports
        "top": 1000,
        # Port version detection: 
        # -sV - Enables version detection
        # -sC - Port scan + default sets of scripts, i.e. --script=default
        "options": [
            # "-sV",
            "-sC"
        ]
    },
    "web_scan":
    {
        "port": 80,
        "subscans": [
            "basic",
            "dirs",
            # "subdomains",
            "waf"
        ],
        "options" :  {
            "subdomains" : {},
            "waf" : {},
            "dirs": {
                "wordlists": [
                    "Discovery/Web-Content/common.txt"
                ],
                "ext": ["php", "html", "zip", "conf", "bak", "txt", ""],
                "skip_codes": [404, 301, 403]
            }
        },
        
    },
}

In [ ]:
# Basic scan
if "basic" in config["scope"]:
    basic_host_scan(config)

In [ ]:
# Nmap
if "ports" in config["scope"]:
    port_scan(config)

In [ ]:
# Dirscan
if "web_scan" in config["scope"]:
    web_scan(config)

In [ ]:
import os

ip = "10.10.11.227"
host = "tickets.keeper.htb"
ROOT_DIR = os.path.dirname(os.path.abspath(""))

wlists = [
    # ROOT_DIR + "/notebook/assets/wordlists/Discovery/Web-Content/common.txt",
    ROOT_DIR + "/notebook/assets/wordlists/Discovery/Web-Content/directory-list-2.3-medium.txt"
]

subdomains = [
    ROOT_DIR + "/notebook/assets/wordlists/Discovery/DNS/subdomains-top1million-20000.txt",
]

cmds = []

delimeter = "\n"
# delimeter = " && "

print(f"sudo sh -c 'echo \"{ip} {host}\" >> /etc/hosts'")

for w in wlists:
    # Nmap
    cmds.append(f"sudo nmap -sC --top-ports 1000 {host}")
    cmds.append(f"sudo nmap -sV --top-ports 1000 {host}")
    # DirSearch
    cmds.append(f"dirsearch -u http://{host} -t 50")
    cmds.append(f"dirsearch -u http://{host} -w {w} -t 50")
    # FFUF
    cmds.append(f"ffuf -w {w}:FUZZ -u http://{host}/FUZZ")
    cmds.append(f"ffuf -w {w}:FUZZ -u http://{host}/FUZZ -e .php,.txt")

for sd in subdomains:
    cmds.append(f"ffuf -w {sd}:FUZZ -u http://{host} -H \"Host: FUZZ.{host}\" -fs 154") # -fs <response size>

print(delimeter.join(cmds))

